# Vietnamese Meeting Diarization + NghiASR (Google Colab)

This notebook implements a **traditional speaker diarization → ASR** pipeline for Vietnamese meeting audio:

1. Upload or point to an audio file.
2. Convert it with **FFmpeg** to **mono, 16 kHz, 16-bit PCM WAV**.
3. Optionally apply moderate loudness normalization.
4. Run **pyannote `speaker-diarization-community-1`** on the **whole recording**.
5. Remove diarization turns shorter than **2.0 seconds**.
6. Run **`NghiMe/NghiASR`** ASR (through Sherpa-ONNX) on the retained turns.
7. Export speaker-attributed transcript files.

### Why this order?

Diarization is run on the full meeting before the `< 2 s` filter. This lets the model use the whole recording for global speaker clustering. The short-turn filter is then applied to the diarization result before ASR.

### Models

- **Diarization:** `pyannote/speaker-diarization-community-1`
- **Vietnamese ASR:** `NghiMe/NghiASR`
  - The notebook uses the recommended `epoch-4-avg-4.int8` ONNX checkpoint through Sherpa-ONNX.

> **Important:** `pyannote/speaker-diarization-community-1` is gated on Hugging Face. You need to accept its conditions once and provide an HF token for the initial download. Inference itself then runs locally in Colab.


## 1. Runtime setup

For diarization, choose **Runtime → Change runtime type → T4 GPU** (or another GPU) in Colab.

NghiASR is run through Sherpa-ONNX using its INT8 ONNX checkpoint. In this notebook, ASR runs on CPU so the Colab GPU remains available for pyannote diarization.


In [ ]:
# System + Python dependencies
!apt-get -qq update
!apt-get -qq install -y ffmpeg
%pip -q install -U "pyannote.audio>=4.0,<5.0" "huggingface_hub>=0.34" \
    sherpa-onnx soundfile pandas tqdm matplotlib

> After the installation cell, Colab may occasionally ask you to restart the runtime because audio/ML packages can replace preinstalled dependencies. If it does, restart and continue from the next cell.

In [ ]:
import os
import sys
import json
import math
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

Set `INPUT_AUDIO` to the file you upload into the Colab filesystem.

The default short-turn rule is exactly the requested behavior:

```python
MIN_SEGMENT_SEC = 2.0
```

Any diarization turn shorter than 2 seconds is discarded **before ASR**.

In [ ]:
# ---------- User configuration ----------

# Example after uploading ZOOM0151.MP3 to /content:
INPUT_AUDIO = "/content/ZOOM0151.MP3"

WORK_DIR = Path("/content/meeting_diarization")
WORK_DIR.mkdir(parents=True, exist_ok=True)

NORMALIZED_WAV = WORK_DIR / "audio_16k_mono.wav"

# Requested noise rule:
MIN_SEGMENT_SEC = 2.0

# FFmpeg loudness normalization.
# Set False if you want only format conversion (mono + 16 kHz + PCM16).
USE_LOUDNORM = True

# Optional speaker-count constraints.
# Leave all as None for fully automatic speaker counting.
NUM_SPEAKERS = None       # e.g. 4
MIN_SPEAKERS = None       # e.g. 2
MAX_SPEAKERS = None       # e.g. 8

# Tiny context pad for ASR around each retained diarization turn.
# Keep small so we do not pull much speech from a neighboring speaker.
ASR_PAD_SEC = 0.05

# CPU threads used by Sherpa-ONNX NghiASR.
ASR_NUM_THREADS = max(1, min(4, os.cpu_count() or 1))

print("INPUT_AUDIO:", INPUT_AUDIO)
print("WORK_DIR:", WORK_DIR)


## 3. Normalize audio with FFmpeg

The output is:

- mono
- 16 kHz
- 16-bit PCM WAV

When `USE_LOUDNORM=True`, FFmpeg also applies a moderate EBU R128 loudness normalization. This is useful for meetings with uneven recording levels, but it does **not** perform aggressive denoising or source separation, which could damage speaker characteristics used by diarization.

In [ ]:
def normalize_audio_ffmpeg(
    input_path,
    output_path,
    sample_rate=16000,
    use_loudnorm=True,
):
    input_path = str(input_path)
    output_path = str(output_path)

    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Audio file not found: {input_path}\n"
            "Upload the file to Colab or change INPUT_AUDIO."
        )

    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-vn",
        "-ac", "1",
        "-ar", str(sample_rate),
    ]

    if use_loudnorm:
        # Moderate single-pass EBU R128 normalization.
        cmd += ["-af", "loudnorm=I=-23:LRA=11:TP=-2"]

    cmd += [
        "-c:a", "pcm_s16le",
        output_path,
    ]

    print("Running:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

normalize_audio_ffmpeg(
    INPUT_AUDIO,
    NORMALIZED_WAV,
    sample_rate=16000,
    use_loudnorm=USE_LOUDNORM,
)

info = sf.info(str(NORMALIZED_WAV))
print("\nNormalized audio:")
print(info)

In [ ]:
# Optional sanity check / playback
from IPython.display import Audio, display

display(Audio(str(NORMALIZED_WAV)))

## 4. Hugging Face access for pyannote

Before running this section:

1. Open the Hugging Face page for `pyannote/speaker-diarization-community-1`.
2. Accept the model's user conditions.
3. Create a Hugging Face read token.
4. Either:
   - add it in **Colab → Secrets** under the name `HF_TOKEN`, or
   - enter it securely when prompted below.

The token is used to download the model. The audio itself is processed locally by the Colab runtime.

In [ ]:
from getpass import getpass

HF_TOKEN = None

# Preferred: Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

# Fallback: secure prompt
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter Hugging Face token: ")

if not HF_TOKEN:
    raise ValueError("A Hugging Face token is required for the gated pyannote model.")

print("HF token loaded.")

## 5. Load the local diarization pipeline

In [ ]:
from pyannote.audio import Pipeline

DIARIZATION_MODEL = "pyannote/speaker-diarization-community-1"

pipeline = Pipeline.from_pretrained(
    DIARIZATION_MODEL,
    token=HF_TOKEN,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline.to(device)

print("Diarization model:", DIARIZATION_MODEL)
print("Diarization device:", device)

## 6. Run diarization on the whole meeting

If you already know the exact number of real meeting participants, setting `NUM_SPEAKERS` can improve stability. Otherwise leave it as `None`.

You can also provide only `MIN_SPEAKERS` / `MAX_SPEAKERS` if you know a plausible range.

In [ ]:
from pyannote.audio.pipelines.utils.hook import ProgressHook

diar_kwargs = {}

if NUM_SPEAKERS is not None:
    diar_kwargs["num_speakers"] = int(NUM_SPEAKERS)
else:
    if MIN_SPEAKERS is not None:
        diar_kwargs["min_speakers"] = int(MIN_SPEAKERS)
    if MAX_SPEAKERS is not None:
        diar_kwargs["max_speakers"] = int(MAX_SPEAKERS)

print("Diarization constraints:", diar_kwargs if diar_kwargs else "automatic")

with ProgressHook() as hook:
    diar_output = pipeline(
        str(NORMALIZED_WAV),
        hook=hook,
        **diar_kwargs,
    )

speaker_diarization = diar_output.speaker_diarization

print("Diarization complete.")

## 7. Convert diarization to a table and remove `< 2 s` turns

This filter does **not** change the speaker clustering that was already computed. It simply prevents short diarization turns from being sent to ASR or included in the final transcript.

In [ ]:
rows = []

for segment, track, speaker in speaker_diarization.itertracks(yield_label=True):
    start = float(segment.start)
    end = float(segment.end)
    duration = end - start

    rows.append({
        "start": start,
        "end": end,
        "duration": duration,
        "speaker": str(speaker),
    })

diar_df = (
    pd.DataFrame(rows)
    .sort_values(["start", "end"])
    .reset_index(drop=True)
)

if diar_df.empty:
    raise RuntimeError("No speech segments were returned by diarization.")

filtered_df = (
    diar_df[diar_df["duration"] >= MIN_SEGMENT_SEC]
    .copy()
    .reset_index(drop=True)
)

removed_df = (
    diar_df[diar_df["duration"] < MIN_SEGMENT_SEC]
    .copy()
    .reset_index(drop=True)
)

if filtered_df.empty:
    raise RuntimeError(
        f"No diarization turns survived the {MIN_SEGMENT_SEC:.1f}s minimum-duration filter. "
        "Lower MIN_SEGMENT_SEC or inspect diarization_raw.csv."
    )

print(f"Raw diarization turns:       {len(diar_df)}")
print(f"Retained >= {MIN_SEGMENT_SEC:.1f}s:       {len(filtered_df)}")
print(f"Removed  < {MIN_SEGMENT_SEC:.1f}s:       {len(removed_df)}")
print(f"Detected speakers (raw):     {diar_df['speaker'].nunique()}")
print(f"Speakers after turn filter:  {filtered_df['speaker'].nunique()}")

display(filtered_df.head(20))

In [ ]:
# Save both raw and filtered diarization tables.
raw_diar_csv = WORK_DIR / "diarization_raw.csv"
filtered_diar_csv = WORK_DIR / "diarization_filtered_ge_2s.csv"
removed_diar_csv = WORK_DIR / "diarization_removed_lt_2s.csv"
raw_rttm = WORK_DIR / "diarization_raw.rttm"

diar_df.to_csv(raw_diar_csv, index=False)
filtered_df.to_csv(filtered_diar_csv, index=False)
removed_df.to_csv(removed_diar_csv, index=False)

with open(raw_rttm, "w", encoding="utf-8") as f:
    speaker_diarization.write_rttm(f)

print("Saved:")
print(raw_diar_csv)
print(filtered_diar_csv)
print(removed_diar_csv)
print(raw_rttm)

### Optional diarization timeline

This is only a quick visual check. Each horizontal row corresponds to a predicted speaker.

In [ ]:
import matplotlib.pyplot as plt

speakers = sorted(diar_df["speaker"].unique())
speaker_to_y = {s: i for i, s in enumerate(speakers)}

plt.figure(figsize=(16, max(3, 0.55 * len(speakers))))
for _, row in diar_df.iterrows():
    y = speaker_to_y[row["speaker"]]
    plt.hlines(
        y=y,
        xmin=row["start"],
        xmax=row["end"],
        linewidth=6,
    )

plt.yticks(range(len(speakers)), speakers)
plt.xlabel("Time (seconds)")
plt.ylabel("Predicted speaker")
plt.title("Raw diarization timeline")
plt.grid(axis="x", alpha=0.25)
plt.show()

In [ ]:
speakers = sorted(diar_df["speaker"].unique())
speaker_to_y = {s: i for i, s in enumerate(speakers)}

plt.figure(figsize=(16, max(3, 0.55 * len(speakers))))
for _, row in filtered_df.iterrows():
    y = speaker_to_y[row["speaker"]]
    plt.hlines(
        y=y,
        xmin=row["start"],
        xmax=row["end"],
        linewidth=6,
    )

plt.yticks(range(len(speakers)), speakers)
plt.xlabel("Time (seconds)")
plt.ylabel("Predicted speaker")
plt.title("Filtered diarization timeline")
plt.grid(axis="x", alpha=0.25)
plt.show()

## 8. Download `NghiMe/NghiASR`

We use the model's recommended **`epoch-4-avg-4.int8`** checkpoint:

- `encoder-epoch-4-avg-4.int8.onnx`
- `decoder-epoch-4-avg-4.int8.onnx`
- `joiner-epoch-4-avg-4.int8.onnx`
- `tokens.txt`

These files are downloaded directly from the Hugging Face repository `NghiMe/NghiASR` and loaded with Sherpa-ONNX.


In [ ]:
from huggingface_hub import hf_hub_download

ASR_REPO_ID = "NghiMe/NghiASR"
ASR_DIR = WORK_DIR / "NghiASR"
ASR_DIR.mkdir(parents=True, exist_ok=True)

ASR_FILES = [
    "encoder-epoch-4-avg-4.int8.onnx",
    "decoder-epoch-4-avg-4.int8.onnx",
    "joiner-epoch-4-avg-4.int8.onnx",
    "tokens.txt",
]

downloaded_asr_files = {}
for filename in ASR_FILES:
    print(f"Downloading {filename} ...")
    cached_path = hf_hub_download(repo_id=ASR_REPO_ID, filename=filename)
    target_path = ASR_DIR / filename
    if not target_path.exists():
        shutil.copy2(cached_path, target_path)
    downloaded_asr_files[filename] = target_path

print("\nNghiASR model directory:", ASR_DIR)
for filename, path in downloaded_asr_files.items():
    print(" -", filename, "->", path)


## 9. Create the NghiASR Sherpa-ONNX recognizer

`NghiMe/NghiASR` is a Zipformer transducer exported to ONNX, so we load its encoder, decoder, joiner, and token table with Sherpa-ONNX.


In [ ]:
import sherpa_onnx

tokens = ASR_DIR / "tokens.txt"
encoder = ASR_DIR / "encoder-epoch-4-avg-4.int8.onnx"
decoder = ASR_DIR / "decoder-epoch-4-avg-4.int8.onnx"
joiner = ASR_DIR / "joiner-epoch-4-avg-4.int8.onnx"

for required in [tokens, encoder, decoder, joiner]:
    if not required.exists():
        raise FileNotFoundError(f"Missing NghiASR model file: {required}")

recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    tokens=str(tokens),
    encoder=str(encoder),
    decoder=str(decoder),
    joiner=str(joiner),
    num_threads=ASR_NUM_THREADS,
    sample_rate=16000,
    feature_dim=80,
    decoding_method="greedy_search",
)

print("NghiASR recognizer ready.")


## 10. Run ASR on the retained diarization turns

The normalized WAV is loaded once into memory, then each retained diarization turn is sliced and passed to NghiASR.

This first notebook intentionally uses a simple **turn-level diarization → ASR** design. Later, we can improve alignment by running ASR/VAD at finer granularity and reconciling word timestamps with speaker turns.


In [ ]:
audio, sample_rate = sf.read(
    str(NORMALIZED_WAV),
    dtype="float32",
    always_2d=False,
)

if audio.ndim != 1:
    raise ValueError(f"Expected mono audio, got shape {audio.shape}")
if sample_rate != 16000:
    raise ValueError(f"Expected 16 kHz audio, got {sample_rate} Hz")

audio_duration = len(audio) / sample_rate
print(f"Loaded {audio_duration/60:.2f} minutes of normalized audio.")

In [ ]:
from tqdm.auto import tqdm

def transcribe_waveform(samples: np.ndarray, sample_rate: int = 16000) -> str:
    samples = np.asarray(samples, dtype=np.float32)

    if samples.size == 0:
        return ""

    stream = recognizer.create_stream()
    stream.accept_waveform(sample_rate, samples)
    recognizer.decode_stream(stream)

    text = stream.result.text
    return text.strip() if text else ""

asr_rows = []

for _, row in tqdm(
    filtered_df.iterrows(),
    total=len(filtered_df),
    desc="NghiASR",
):
    start = float(row["start"])
    end = float(row["end"])

    padded_start = max(0.0, start - ASR_PAD_SEC)
    padded_end = min(audio_duration, end + ASR_PAD_SEC)

    i0 = int(round(padded_start * sample_rate))
    i1 = int(round(padded_end * sample_rate))

    segment_audio = audio[i0:i1]
    text = transcribe_waveform(segment_audio, sample_rate)

    asr_rows.append({
        "start": start,
        "end": end,
        "duration": float(row["duration"]),
        "speaker": row["speaker"],
        "text": text,
    })

transcript_df = pd.DataFrame(asr_rows)

display(transcript_df.head(30))


## 11. Build and export the speaker-attributed transcript

In [ ]:
def format_timestamp(seconds: float) -> str:
    milliseconds = int(round(seconds * 1000))
    hours, rem = divmod(milliseconds, 3_600_000)
    minutes, rem = divmod(rem, 60_000)
    secs, ms = divmod(rem, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{ms:03d}"

transcript_lines = []

for _, row in transcript_df.iterrows():
    start_ts = format_timestamp(float(row["start"]))
    end_ts = format_timestamp(float(row["end"]))
    speaker = row["speaker"]
    text = str(row["text"]).strip()

    transcript_lines.append(
        f"[{start_ts} - {end_ts}] {speaker}: {text}"
    )

transcript_text = "\n".join(transcript_lines)

print(transcript_text[:10000])
if len(transcript_text) > 10000:
    print("\n... output truncated in notebook display ...")

In [ ]:
transcript_csv = WORK_DIR / "speaker_transcript.csv"
transcript_json = WORK_DIR / "speaker_transcript.json"
transcript_txt = WORK_DIR / "speaker_transcript.txt"

transcript_df.to_csv(transcript_csv, index=False)

with open(transcript_json, "w", encoding="utf-8") as f:
    json.dump(
        transcript_df.to_dict(orient="records"),
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(transcript_txt, "w", encoding="utf-8") as f:
    f.write(transcript_text)

print("Saved final outputs:")
print(transcript_csv)
print(transcript_json)
print(transcript_txt)

## 12. Quick diagnostics

This helps catch obvious failure modes before moving on to meeting summarization.

In [ ]:
summary = {
    "audio_minutes": round(audio_duration / 60, 2),
    "raw_turns": int(len(diar_df)),
    "retained_turns_ge_2s": int(len(filtered_df)),
    "removed_turns_lt_2s": int(len(removed_df)),
    "raw_detected_speakers": int(diar_df["speaker"].nunique()),
    "retained_speakers": int(filtered_df["speaker"].nunique()),
    "retained_speech_minutes": round(filtered_df["duration"].sum() / 60, 2),
    "discarded_short_turn_seconds": round(removed_df["duration"].sum(), 2),
    "empty_asr_turns": int((transcript_df["text"].str.len() == 0).sum()),
}

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

## 13. Optional: download the result files

In [ ]:
from google.colab import files

# Uncomment any files you want to download:
files.download(str(transcript_txt))
# files.download(str(transcript_csv))
# files.download(str(transcript_json))
# files.download(str(filtered_diar_csv))
# files.download(str(raw_rttm))

# Notes / next improvements

This notebook deliberately starts with the **traditional diarization pipeline**.

Useful next steps for a production meeting-note system would be:

- distinguish **main participants** from people who only chime in;
- merge or suppress very fragmented speaker turns more intelligently than a hard duration rule;
- use ASR timestamps and pyannote's exclusive diarization output for tighter alignment;
- add speaker enrollment only if known-participant identity is needed;
- evaluate DER/JER for diarization and WER/CER for ASR on a small labeled set of your actual meeting recordings;
- optionally add a dedicated noise/VAD stage, but only after checking that it does not damage speaker embeddings.